In [2]:
import os
import pickle
from PIL import Image
import torch
from facenet_pytorch import InceptionResnetV1
from torchvision import transforms
from scipy.spatial.distance import cosine
import numpy as np

In [3]:
# Inisialisasi model FaceNet
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = InceptionResnetV1(pretrained='vggface2').eval().to(device)

In [4]:
# Fungsi untuk load gambar dan preprocessing
def preprocess_image(image_path):
    img = Image.open(image_path).convert('RGB')
    preprocess = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    return preprocess(img).unsqueeze(0).to(device)

In [ ]:
# Load embedding training dari file pickle
with open('combined_embeddings.pkl', 'rb') as f:
    train_embeddings = pickle.load(f)

In [6]:
# Fungsi untuk melakukan prediksi pada folder tertentu
def predict_folder(folder_path):
    true_labels = []
    predicted_labels = []

    for label in os.listdir(folder_path):
        label_folder = os.path.join(folder_path, label)

        if os.path.isdir(label_folder):
            for img_file in os.listdir(label_folder):
                if img_file.endswith('.ppm'):
                    img_path = os.path.join(label_folder, img_file)

                    # Preprocess gambar
                    img_tensor = preprocess_image(img_path)

                    # Dapatkan embedding untuk gambar
                    with torch.no_grad():
                        embedding = model(img_tensor).cpu().numpy().flatten()

                    # Simpan label sebenarnya
                    true_labels.append(label)

                    # Cari embedding training terdekat berdasarkan cosine similarity
                    best_label = None
                    min_distance = float('inf')

                    for train_label, embeddings in train_embeddings.items():
                        for train_embedding in embeddings:
                            # Hitung jarak cosine
                            distance = cosine(embedding, train_embedding)
                            if distance < min_distance:
                                min_distance = distance
                                best_label = train_label

                    # Simpan prediksi label
                    predicted_labels.append(best_label)

                    # Print hasil prediksi
                    print(f"{img_file}({label}) => {best_label}")

    # Menghitung akurasi
    correct_predictions = sum([1 for true, pred in zip(true_labels, predicted_labels) if true == pred])
    accuracy = correct_predictions / len(true_labels)

    print(f"\nAccuracy for '{folder_path}': {accuracy * 100:.2f}%")
    return accuracy

In [7]:
# Path folder validasi
val_path = 'val'
train_path = 'train'

In [8]:
# Prediksi pada folder validasi
print("Validation Predictions:")
val_accuracy = predict_folder(val_path)

Validation Predictions:
S001-07-t10_01.ppm(S001) => S001
S001-08-t10_01.ppm(S001) => S001
S001-08-t10_03.ppm(S001) => S001
S006-01-t10_01.ppm(S006) => S006
S008-04-t10_01.ppm(S008) => S008
S013-01-t10_01.ppm(S013) => S013
S015-03-t10_01.ppm(S015) => S015
S022-02-t10_01.ppm(S022) => S331
S023-01-t10_01.ppm(S023) => S023
S023-02-t10_01.ppm(S023) => S023
S027-01-t10_02.ppm(S027) => S027
S027-01-t10_03.ppm(S027) => S027
S029-01-t10_01.ppm(S029) => S029
S029-02-t10_01.ppm(S029) => S029
S030-01-t10_01.ppm(S030) => S030
S030-03-t10_01.ppm(S030) => S030
S031-01-t10_01.ppm(S031) => S008
S031-02-t10_01.ppm(S031) => S031
S032-01-t10_01.ppm(S032) => S032
S033-04-t10_01.ppm(S033) => S033
S036-01-t10_01.ppm(S036) => S036
S044-04-t10_01.ppm(S044) => S044
S044-05-t10_01.ppm(S044) => S044
S045-01-t10_01.ppm(S045) => S045
S046-01-t10_01.ppm(S046) => S046
S048-03-t10_01.ppm(S048) => S048
S048-05-t10_01.ppm(S048) => S048
S051-01-t10_01.ppm(S051) => S051
S052-01-t10_01.ppm(S052) => S052
S052-03-t10_01.ppm(

In [9]:
# Prediksi pada folder train untuk melihat overfitting
print("\nTraining Predictions:")
train_accuracy = predict_folder(train_path)


Training Predictions:
S001-01-t10_01.ppm(S001) => S001
S001-02-t10_01.ppm(S001) => S001
S001-03-t10_01.ppm(S001) => S001
S001-04-t10_01.ppm(S001) => S001
S001-05-t10_01.ppm(S001) => S001
S001-06-t10_01.ppm(S001) => S001
S001-08-t10_02.ppm(S001) => S001
S001-09-t10_01.ppm(S001) => S001
S006-02-t10_01.ppm(S006) => S006
S006-03-t10_01.ppm(S006) => S006
S008-01-t10_01.ppm(S008) => S008
S008-02-t10_01.ppm(S008) => S008
S008-03-t10_01.ppm(S008) => S008
S008-05-t10_01.ppm(S008) => S008
S008-06-t10_01.ppm(S008) => S008
S008-07-t10_01.ppm(S008) => S008
S013-02-t10_01.ppm(S013) => S013
S015-01-t10_01.ppm(S015) => S015
S015-02-t10_01.ppm(S015) => S015
S022-01-t10_01.ppm(S022) => S022
S023-03-t10_01.ppm(S023) => S023
S023-04-t10_01.ppm(S023) => S023
S027-01-t10_01.ppm(S027) => S027
S029-03-t10_01.ppm(S029) => S029
S030-02-t10_01.ppm(S030) => S030
S030-04-t10_01.ppm(S030) => S030
S030-05-t10_01.ppm(S030) => S030
S031-03-t10_01.ppm(S031) => S031
S031-04-t10_01.ppm(S031) => S031
S031-05-t10_01.ppm(S